# 🚀 ys_LoRA_Training - Fine-Tuning LoRA Descomplicado & Modular

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yashikiDEV/ys_loRA_Training/blob/main/ys_LoRA_Training.ipynb)
[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](https://opensource.org/licenses/MIT)

Bem-vindo ao **ys_LoRA_Training**! Uma ferramenta aberta, intuitiva e modular para qualquer pessoa realizar **fine-tuning** de Modelos de Linguagem de Grande Porte (LLMs como **Qwen**, **Llama**, **Mistral**, **Gemma**, **Phi**, **GPT-2**, etc.) usando a técnica **LoRA / QLoRA** diretamente no **Google Colab**.

---

### 💡 Por que este notebook é diferente?
- **Configuração Centralizada**: Você edita apenas **UMA** célula (a Célula 2). Nada de caçar variáveis ou caminhos espalhados pelo código.
- **Zero Exemplos Embutidos**: Nenhum dado de treino fica fixo no código. Seu dataset é lido externamente de arquivos limpos (`.json`, `.jsonl` ou `.txt`).
- **Compatível com Qualquer Arquitetura**: Detecta automaticamente as camadas de atenção e projeção do modelo (`TARGET_MODULES = "auto"`).
- **Amigável ao Colab Gratuito (GPU T4)**: Suporta quantização 4-bit (QLoRA) e Gradient Checkpointing para caber na VRAM de 15 GB sem estourar memória.
- **Segurança no Google Drive**: Seus adaptadores, checkpoints e relatórios (`info_treino.json`) são salvos direto no seu Google Drive com versionamento automático (`v0`, `v1`, `v2`...).

---

### ⚡ Como usar em 5 passos:
1. **Ative a GPU no Colab**: Vá no menu superior em `Ambiente de execução` -> `Alterar tipo de ambiente de execução` -> selecione **T4 GPU** (ou superior) e clique em **Salvar**.
2. **Execute a Instalação (Célula 1)**: Remove conflitos do `torchao` e instala as bibliotecas necessárias.
3. **Preencha a Configuração (Célula 2)**: Altere o modelo desejado, caminhos do Drive, dataset e hiperparâmetros pelo formulário visual.
4. **Execute as células em sequência (3 a 9)**: Conecte o Drive, valide os dados, carregue o modelo e inicie o treino.
5. **Teste seu modelo (Célula 10)**: Converse em tempo real com o seu modelo treinado!

📖 Para documentação completa, guia de datasets e dicas avançadas, consulte o [README.md no repositório GitHub](https://github.com/yashikiDEV/ys_loRA_Training).


## 📦 1. Instalação das Dependências

Esta célula desinstala o pacote `torchao` (prevenindo conflitos de ABI e compatibilidade com `bitsandbytes` no ambiente do Colab) e instala as versões estáveis das bibliotecas essenciais para LoRA/QLoRA.


In [ ]:
# 1. Desinstalar torchao para prevenir incompatibilidade com bitsandbytes e PyTorch no Colab
!pip uninstall -y torchao

# 2. Instalar bibliotecas essenciais para LoRA / QLoRA
!pip install --quiet --upgrade \
    "transformers>=4.44.0" \
    "peft>=0.12.0" \
    "datasets>=2.20.0" \
    "accelerate>=0.33.0" \
    "bitsandbytes>=0.43.0" \
    "scipy" \
    "huggingface_hub"

print("\n" + "="*50)
print("✅ Todas as dependências foram instaladas com sucesso!")
print("="*50)


## 🛠️ 2. CONFIGURAÇÃO (Única célula que você precisa editar)

Utilize os campos de formulário abaixo para definir todas as opções do seu treinamento. Todas as variáveis do notebook se originam exclusivamente desta célula.


In [ ]:
# @title ⚙️ Painel de Configuração do Treinamento
# @markdown ---
# @markdown ### 🤖 1. Modelo & Autenticação
# @markdown Nome do modelo no Hugging Face (ex.: `Qwen/Qwen2.5-0.5B`, `meta-llama/Llama-3.2-1B`, `google/gemma-2-2b-it`) ou caminho no Drive:
MODEL_NAME_OR_PATH = "Qwen/Qwen2.5-0.5B"  # @param {type:"string"}

# @markdown Nome da chave nos Segredos do Colab (Colab Secrets) contendo o Token do Hugging Face (opcional, para modelos fechados):
HF_TOKEN_SECRET_NAME = "HF_TOKEN"  # @param {type:"string"}

# @markdown Ou cole o Token do Hugging Face manualmente aqui caso prefira não usar os Secrets (deixe vazio se o modelo for público):
HF_TOKEN_MANUAL = ""  # @param {type:"string"}

# @markdown ---
# @markdown ### 📁 2. Caminhos & Projeto no Google Drive
# @markdown Nome do seu projeto (será o nome da pasta organizada no Drive):
PROJECT_NAME = "meu_projeto_lora"  # @param {type:"string"}

# @markdown Caminho da pasta base no Google Drive onde os projetos serão salvos:
BASE_DIR = "/content/drive/MyDrive/meu_projeto_lora"  # @param {type:"string"}

# @markdown Caminho do arquivo de dataset (no Google Drive ou caminho local):
DATASET_PATH = "/content/drive/MyDrive/meu_projeto_lora/exemplo_dataset.json"  # @param {type:"string"}

# @markdown Formato do dataset ('auto' detecta pela extensão .json, .jsonl ou .txt):
DATASET_FORMAT = "auto"  # @param ["auto", "jsonl", "json", "txt"]

# @markdown Nome da coluna/campo de texto caso o dataset seja no formato texto livre:
TEXT_FIELD_NAME = "text"  # @param {type:"string"}

# @markdown Nome do campo de mensagens caso o dataset seja no formato de conversação:
MESSAGES_FIELD_NAME = "messages"  # @param {type:"string"}

# @markdown ---
# @markdown ### 💬 3. Template de Prompt & Tokens Especiais
# @markdown Prefixo que identifica o início da mensagem do usuário:
USER_TOKEN = "<|usuario|>\n"  # @param {type:"string"}

# @markdown Prefixo que identifica o início da resposta do assistente:
ASSISTANT_TOKEN = "\n<|assistente|>\n"  # @param {type:"string"}

# @markdown Token de término da resposta. Deixe vazio ("") para usar automaticamente o eos_token do tokenizer:
END_TOKEN = ""  # @param {type:"string"}

# @markdown Instrução de sistema inicial opcional (deixe vazio se não quiser usar):
SYSTEM_PROMPT = ""  # @param {type:"string"}

# @markdown ---
# @markdown ### 🎛️ 4. Hiperparâmetros de Treinamento
# @markdown Quantidade de passagens completas pelos dados (épocas):
NUM_EPOCHS = 3  # @param {type:"integer"}

# @markdown Taxa de aprendizado inicial (learning rate):
LEARNING_RATE = 2e-4  # @param {type:"number"}

# @markdown Tamanho do lote (batch size) por passo na GPU (reduza para 1 se faltar memória):
BATCH_SIZE = 2  # @param {type:"integer"}

# @markdown Passos de acumulação de gradiente (batch efetivo = BATCH_SIZE * GRAD_ACCUM_STEPS):
GRAD_ACCUM_STEPS = 4  # @param {type:"integer"}

# @markdown Tamanho máximo de tokens por exemplo (reduza para 256 ou 512 se a GPU for limitada):
MAX_LENGTH = 512  # @param {type:"integer"}

# @markdown Proporção inicial do treino usada para aquecer o learning rate (warmup):
WARMUP_RATIO = 0.05  # @param {type:"number"}

# @markdown Regularização por decaimento de peso (weight decay):
WEIGHT_DECAY = 0.01  # @param {type:"number"}

# @markdown Semente para reprodutibilidade dos resultados (random seed):
RANDOM_SEED = 42  # @param {type:"integer"}

# @markdown Frequência de exibição dos logs no treino (a cada N passos):
LOGGING_STEPS = 5  # @param {type:"integer"}

# @markdown Salvar checkpoints intermediários a cada N passos (0 para salvar apenas no final):
SAVE_STEPS = 50  # @param {type:"integer"}

# @markdown ---
# @markdown ### 🎯 5. Configurações do LoRA (PEFT)
# @markdown Dimensão dos adaptadores LoRA (Rank `r`). Valores usuais: 8, 16, 32, 64:
LORA_R = 16  # @param {type:"integer"}

# @markdown Fator de escala `alpha` (geralmente 2x o valor de `r`):
LORA_ALPHA = 32  # @param {type:"integer"}

# @markdown Dropout para regularização do LoRA:
LORA_DROPOUT = 0.05  # @param {type:"number"}

# @markdown Módulos-alvo para LoRA. Use "auto" para detectar todas as camadas lineares automaticamente ou liste-as separadas por vírgula (ex.: "q_proj,v_proj"):
TARGET_MODULES = "auto"  # @param {type:"string"}

# @markdown ---
# @markdown ### ⚡ 6. Quantização & Otimização de Memória (VRAM)
# @markdown Tipo de quantização (4-bit é altamente recomendado para a GPU T4 gratuita do Colab):
QUANTIZATION = "4-bit"  # @param ["none", "4-bit", "8-bit"]

# @markdown Ativar Gradient Checkpointing (economiza MUITA memória de GPU):
GRADIENT_CHECKPOINTING = True  # @param {type:"boolean"}

# @markdown ---
# @markdown ### 🏷️ 7. Versionamento & Continuidade
# @markdown Modo de versionamento das pastas de treino:
VERSIONING_MODE = "auto"  # @param ["auto", "manual"]

# @markdown Nome da versão caso o modo acima seja "manual" (ex.: "v0", "experimento_1"):
MANUAL_VERSION_NAME = "v0"  # @param {type:"string"}

# @markdown Retomar treinamento a partir de um checkpoint salvo anteriormente:
RESUME_TRAINING = False  # @param {type:"boolean"}

# @markdown Nome da versão de onde retomar o treinamento (caso RESUME_TRAINING seja True):
RESUME_VERSION_NAME = "v0"  # @param {type:"string"}

# @markdown ---
# @markdown ### 🔀 8. Fusão (Merge) do Modelo
# @markdown Fundir (merge) os pesos do adaptador LoRA de volta no modelo base ao final:
MERGE_LORA = False  # @param {type:"boolean"}

print("="*50)
print("✅ Configurações registradas com sucesso!")
print(f"🤖 Modelo Base:     {MODEL_NAME_OR_PATH}")
print(f"📁 Pasta do Projeto: {PROJECT_NAME}")
print(f"⚡ Quantização:     {QUANTIZATION}")
print("="*50)


## 📂 3. Conectar ao Google Drive e Estruturar Pastas

Monta o Google Drive automaticamente e organiza as pastas necessárias para salvar checkpoints, modelos versionados (`v0`, `v1`, ...) e metadados de treino.


In [ ]:
import os
import re
from pathlib import Path

# 1. Montar Google Drive no Colab
try:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        print("🔗 Montando o Google Drive...")
        drive.mount('/content/drive')
    else:
        print("✅ Google Drive já está conectado.")
except ImportError:
    print("ℹ️ Executando fora do Google Colab (ambiente local). Certifique-se de que BASE_DIR é acessível.")

# 2. Configurar caminhos do projeto
project_root = Path(BASE_DIR)
versions_root = project_root / "versoes"
checkpoints_root = project_root / "checkpoints"

# 3. Determinar o nome da versão atual
if RESUME_TRAINING:
    current_version_name = RESUME_VERSION_NAME.strip()
    if not current_version_name:
        raise ValueError("❌ RESUME_TRAINING está ativo, mas RESUME_VERSION_NAME está em branco na Célula 2!")
else:
    if VERSIONING_MODE == "manual":
        current_version_name = MANUAL_VERSION_NAME.strip() or "v0"
    else:
        # Modo 'auto': encontra a próxima versão sequencial (v0, v1, v2...)
        os.makedirs(versions_root, exist_ok=True)
        existing_versions = [
            d for d in os.listdir(versions_root)
            if (versions_root / d).is_dir() and re.match(r"^v(\d+)$", d)
        ]
        if not existing_versions:
            next_idx = 0
        else:
            indices = [int(re.match(r"^v(\d+)$", d).group(1)) for d in existing_versions]
            next_idx = max(indices) + 1
        current_version_name = f"v{next_idx}"

# 4. Criar pastas necessárias
current_version_dir = versions_root / current_version_name
current_checkpoint_dir = checkpoints_root / current_version_name

os.makedirs(current_version_dir, exist_ok=True)
os.makedirs(current_checkpoint_dir, exist_ok=True)

print("\n" + "="*60)
print(f"📁 Pasta Base do Projeto: {project_root}")
print(f"🏷️  Versão Atual:          {current_version_name}")
print(f"💾 Pasta da Versão:       {current_version_dir}")
print(f"⏱️  Pasta de Checkpoints:  {current_checkpoint_dir}")
print("="*60)


## 🔍 4. Validação da Configuração e Ambiente

Verifica se a GPU está ativa, testa a capacidade de VRAM, detecta se o hardware suporta `bf16` ou `fp16`, confere o acesso ao dataset e valida o token do Hugging Face antes de gastar recursos de processamento.


In [ ]:
import os
import torch
from pathlib import Path

def validate_environment():
    """Valida GPU, arquivos de dataset e credenciais antes de iniciar o processo."""
    print("="*60)
    print("🔍 INICIANDO VALIDAÇÃO DO AMBIENTE E PARÂMETROS...")
    print("="*60)

    # 1. Validação de GPU
    if not torch.cuda.is_available():
        error_msg = (
            "❌ ERRO CRÍTICO: Nenhuma GPU foi detectada!\n"
            "O treinamento de LLMs requer uma GPU ativa.\n\n"
            "👉 Como corrigir no Google Colab:\n"
            "1. No menu superior, clique em 'Ambiente de execução' (Runtime)\n"
            "2. Selecione 'Alterar tipo de ambiente de execução' (Change runtime type)\n"
            "3. Em 'Acelerador de hardware', escolha 'T4 GPU' (gratuito) e clique em Salvar."
        )
        raise RuntimeError(error_msg)

    gpu_name = torch.cuda.get_device_name(0)
    vram_bytes = torch.cuda.get_device_properties(0).total_memory
    vram_gb = vram_bytes / (1024 ** 3)
    major_cc, minor_cc = torch.cuda.get_device_capability(0)

    # Decisão automática entre bf16 e fp16 (Ampere >= 8.0 suporta bf16 nativo; T4 usa fp16)
    use_bf16 = major_cc >= 8
    use_fp16 = not use_bf16

    print(f"🎮 GPU Detectada: {gpu_name}")
    print(f"💾 VRAM Total:    {vram_gb:.2f} GB")
    print(f"⚡ Precisão Ideal: {'bfloat16 (bf16)' if use_bf16 else 'float16 (fp16)'}")

    # Alerta preventivo se o modelo for grande para a VRAM sem quantização
    if vram_gb < 16.0 and QUANTIZATION == "none":
        print(
            "\n⚠️  AVISO DE MEMÓRIA:\n"
            f"Você está usando uma GPU com {vram_gb:.1f} GB de VRAM e QUANTIZATION='none'.\n"
            "Modelos com mais de 1B de parâmetros podem esgotar a memória (CUDA OOM) em 16-bit.\n"
            "💡 Dica: Se ocorrer erro de memória, configure QUANTIZATION='4-bit' na Célula 2.\n"
        )

    # 2. Validação do Token Hugging Face
    hf_token = None
    if HF_TOKEN_MANUAL.strip():
        hf_token = HF_TOKEN_MANUAL.strip()
    else:
        try:
            from google.colab import userdata
            hf_token = userdata.get(HF_TOKEN_SECRET_NAME)
        except Exception:
            hf_token = os.environ.get(HF_TOKEN_SECRET_NAME, None)

    if hf_token:
        try:
            import huggingface_hub
            huggingface_hub.login(token=hf_token, add_to_git_credential=False)
            print("🔑 Hugging Face: Autenticado com sucesso!")
        except Exception as e:
            print(f"⚠️ Aviso ao autenticar no Hugging Face: {e}")
    else:
        print("ℹ️ Hugging Face: Nenhum token configurado (adequado para modelos abertos como Qwen, GPT-2).")

    # 3. Validação do Dataset
    dataset_file = Path(DATASET_PATH)
    if not dataset_file.exists():
        error_dataset = (
            f"❌ ERRO: Arquivo do dataset não encontrado!\n"
            f"Caminho verificado: {DATASET_PATH}\n\n"
            f"👉 Como corrigir:\n"
            f"1. Verifique se o Google Drive foi montado corretamente.\n"
            f"2. Certifique-se de que o arquivo existe e o caminho na Célula 2 está correto.\n"
            f"3. Se quiser fazer um teste rápido, você pode usar o arquivo 'exemplo_dataset.json' incluído no repositório."
        )
        raise FileNotFoundError(error_dataset)

    if dataset_file.stat().st_size == 0:
        raise ValueError(f"❌ O arquivo do dataset em '{DATASET_PATH}' está vazio (0 bytes)!")

    print(f"📄 Dataset Localizado: {DATASET_PATH} ({dataset_file.stat().st_size / 1024:.2f} KB)")
    print("\n✅ Todas as validações foram concluídas com sucesso!")
    print("="*60)

    return {
        "gpu_name": gpu_name,
        "vram_gb": vram_gb,
        "use_bf16": use_bf16,
        "use_fp16": use_fp16,
        "hf_token": hf_token
    }

env_info = validate_environment()


## 🧠 5. Carregar Tokenizer e Modelo Base

Inicializa o tokenizer e o modelo base com a quantização escolhida (4-bit, 8-bit ou sem quantização), tratando automaticamente o `pad_token` e o token de fim.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

def load_tokenizer_and_base_model(model_name_or_path, hf_token, quant_type, use_bf16, use_fp16):
    """Carrega o tokenizer e o modelo base com suporte a quantização e tratamento de tokens."""
    print("="*60)
    print(f"📥 CARREGANDO MODELO E TOKENIZER: {model_name_or_path}")
    print("="*60)

    # 1. Carregar Tokenizer
    tokenizer = AutoTokenizer.from_pretrained(
        model_name_or_path,
        token=hf_token,
        trust_remote_code=True
    )

    added_new_pad = False
    # Garantir que o pad_token esteja configurado
    if tokenizer.pad_token is None:
        if tokenizer.eos_token is not None:
            tokenizer.pad_token = tokenizer.eos_token
            print(f"ℹ️ pad_token configurado como eos_token: '{tokenizer.pad_token}'")
        else:
            tokenizer.add_special_tokens({'pad_token': '[PAD]'})
            added_new_pad = True
            print("ℹ️ pad_token criado como '[PAD]'")

    # Alinhamento à direita obrigatório para modelos causais durante treino
    tokenizer.padding_side = "right"

    # Definir token de fim ativo (usa o configurado ou faz fallback para eos_token)
    if END_TOKEN and END_TOKEN.strip():
        active_end_token = END_TOKEN
    else:
        active_end_token = tokenizer.eos_token
    print(f"🎯 Token de fim ativo: '{active_end_token}'")

    # 2. Configurar Quantização (BitsAndBytes)
    compute_dtype = torch.bfloat16 if use_bf16 else torch.float16
    quant_config = None

    if quant_type == "4-bit":
        print("⚡ Configurando quantização 4-bit (QLoRA NF4)...")
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=compute_dtype,
            bnb_4bit_use_double_quant=True
        )
    elif quant_type == "8-bit":
        print("⚡ Configurando quantização 8-bit...")
        quant_config = BitsAndBytesConfig(load_in_8bit=True)
    else:
        print(f"⚡ Carregando em precisão padrão ({'bf16' if use_bf16 else 'fp16'})...")

    # 3. Carregar Modelo
    model_kwargs = {
        "token": hf_token,
        "trust_remote_code": True,
        "device_map": "auto",
    }

    if quant_config is not None:
        model_kwargs["quantization_config"] = quant_config
    else:
        model_kwargs["torch_dtype"] = compute_dtype

    model = AutoModelForCausalLM.from_pretrained(model_name_or_path, **model_kwargs)

    # Se um pad_token novo foi criado, o embedding precisa acompanhar o tamanho do vocabulário
    if added_new_pad:
        model.resize_token_embeddings(len(tokenizer))

    # 4. Habilitar Gradient Checkpointing e Preparar K-Bit Training
    if quant_type in ["4-bit", "8-bit"]:
        model = prepare_model_for_kbit_training(
            model,
            use_gradient_checkpointing=GRADIENT_CHECKPOINTING
        )
    elif GRADIENT_CHECKPOINTING:
        if hasattr(model, "gradient_checkpointing_enable"):
            model.gradient_checkpointing_enable()
        # Necessário para o gradiente fluir com checkpointing + LoRA (sem quantização)
        if hasattr(model, "enable_input_require_grads"):
            model.enable_input_require_grads()

    # Desabilitar cache KV durante o treino (economiza VRAM e evita conflitos com gradient checkpointing)
    model.config.use_cache = False

    # Exibir parâmetros totais
    total_params = sum(p.numel() for p in model.parameters())
    print(f"📊 Parâmetros Totais do Modelo Base: {total_params:,}")
    print("✅ Tokenizer e Modelo Base carregados com sucesso!")
    print("="*60)

    return tokenizer, model, active_end_token

tokenizer, model, ACTIVE_END_TOKEN = load_tokenizer_and_base_model(
    model_name_or_path=MODEL_NAME_OR_PATH,
    hf_token=env_info["hf_token"],
    quant_type=QUANTIZATION,
    use_bf16=env_info["use_bf16"],
    use_fp16=env_info["use_fp16"]
)


## 📊 6. Carregar e Preparar o Dataset

Lê o arquivo do dataset especificado na configuração (`.json`, `.jsonl` ou `.txt`), aplica o template de prompt configurado, calcula estatísticas reais de tokens e exibe 2 amostras do dataset já formatadas (sem qualquer dado embutido no código).


In [ ]:
import json
from pathlib import Path
from datasets import Dataset

def parse_dataset_file(file_path, file_format, text_field, messages_field, user_tok, asst_tok, end_tok, sys_prompt):
    """Lê arquivos TXT, JSON ou JSONL e converte em lista de textos formatados com o template."""
    path = Path(file_path)
    ext = file_format.lower()
    if ext == "auto":
        ext = path.suffix.lower().lstrip(".")

    formatted_texts = []

    def format_item(item):
        """Converte um item individual em texto com o template configurado."""
        if isinstance(item, str):
            text = item.strip()
            if not text:
                return None
            if not text.endswith(end_tok):
                text += end_tok
            return text

        if isinstance(item, dict):
            # Formato 1: Conversação (messages: [{"role": ..., "content": ...}])
            if messages_field in item and isinstance(item[messages_field], list):
                convo = ""
                if sys_prompt.strip():
                    convo += f"{sys_prompt.strip()}\n"
                for msg in item[messages_field]:
                    role = msg.get("role", "").lower()
                    content = msg.get("content", "").strip()
                    if role in ["user", "human", "usuario"]:
                        convo += f"{user_tok}{content}"
                    elif role in ["assistant", "bot", "assistente"]:
                        convo += f"{asst_tok}{content}{end_tok}"
                    elif role in ["system", "sistema"] and not sys_prompt.strip():
                        convo += f"{content}\n"
                return convo.strip()

            # Formato 2: Instruction / Input / Output
            if "instruction" in item or "prompt" in item:
                instruction = item.get("instruction") or item.get("prompt", "")
                inp = item.get("input", "")
                output = item.get("output") or item.get("response", "")

                user_content = f"{instruction}\n{inp}".strip() if inp else instruction.strip()
                convo = ""
                if sys_prompt.strip():
                    convo += f"{sys_prompt.strip()}\n"
                convo += f"{user_tok}{user_content}{asst_tok}{output.strip()}{end_tok}"
                return convo

            # Formato 3: Coluna de texto livre
            if text_field in item:
                text = str(item[text_field]).strip()
                if not text.endswith(end_tok):
                    text += end_tok
                return text

        return None

    # Leitura conforme extensão
    if ext == "jsonl":
        with open(path, "r", encoding="utf-8") as f:
            for line_num, line in enumerate(f, 1):
                line = line.strip()
                if not line:
                    continue
                try:
                    obj = json.loads(line)
                    txt = format_item(obj)
                    if txt:
                        formatted_texts.append(txt)
                except Exception as e:
                    print(f"⚠️ Aviso na linha {line_num} do JSONL: {e}")

    elif ext == "json":
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
            # Se for dicionário com chave tipo 'data' ou 'train'
            if isinstance(data, dict):
                for possible_key in ["data", "samples", "train", "dataset", "rows"]:
                    if possible_key in data and isinstance(data[possible_key], list):
                        data = data[possible_key]
                        break
            if isinstance(data, list):
                for item in data:
                    txt = format_item(item)
                    if txt:
                        formatted_texts.append(txt)
            elif isinstance(data, dict):
                txt = format_item(data)
                if txt:
                    formatted_texts.append(txt)

    elif ext == "txt":
        with open(path, "r", encoding="utf-8") as f:
            raw_content = f.read()
            # Separadores de documentos suportados
            if "\n---\n" in raw_content:
                parts = raw_content.split("\n---\n")
            elif "\n\n\n" in raw_content:
                parts = raw_content.split("\n\n\n")
            else:
                parts = [raw_content]

            for part in parts:
                txt = format_item(part)
                if txt:
                    formatted_texts.append(txt)
    else:
        raise ValueError(f"❌ Formato de dataset '{ext}' não suportado! Use 'jsonl', 'json' ou 'txt'.")

    return formatted_texts

print("="*60)
print("📊 CARREGANDO E PROCESSANDO DATASET...")
print("="*60)

raw_samples = parse_dataset_file(
    file_path=DATASET_PATH,
    file_format=DATASET_FORMAT,
    text_field=TEXT_FIELD_NAME,
    messages_field=MESSAGES_FIELD_NAME,
    user_tok=USER_TOKEN,
    asst_tok=ASSISTANT_TOKEN,
    end_tok=ACTIVE_END_TOKEN,
    sys_prompt=SYSTEM_PROMPT
)

if not raw_samples:
    raise ValueError("❌ Nenhum exemplo válido foi extraído do dataset! Verifique o caminho e a estrutura do arquivo.")

total_samples = len(raw_samples)
print(f"✅ Total de exemplos carregados: {total_samples}")

# Analisar comprimentos reais de tokens antes de truncar
token_lengths = [len(tokenizer.encode(s)) for s in raw_samples]
min_len = min(token_lengths)
max_len = max(token_lengths)
avg_len = sum(token_lengths) / total_samples
exceeded = sum(1 for l in token_lengths if l > MAX_LENGTH)

print("\n📈 ESTATÍSTICAS DE COMPRIMENTO EM TOKENS:")
print(f"  • Mínimo:       {min_len} tokens")
print(f"  • Médio:        {avg_len:.1f} tokens")
print(f"  • Máximo:       {max_len} tokens")
print(f"  • Limite Atual: {MAX_LENGTH} tokens")
if exceeded > 0:
    print(f"  ⚠️  AVISO: {exceeded} exemplos ({exceeded/total_samples*100:.1f}%) ultrapassam o MAX_LENGTH={MAX_LENGTH} e serão truncados.")
else:
    print("  ✅ Todos os exemplos cabem perfeitamente dentro do MAX_LENGTH configurado!")

# Exibir 2 amostras reais já formatadas
print("\n" + "="*60)
print("👀 AMOSTRAS DO DATASET (Exemplos reais formatados):")
print("="*60)
for idx in range(min(2, total_samples)):
    print(f"\n--- [Exemplo {idx + 1}] ---")
    print(raw_samples[idx])
print("="*60)

# Criar Hugging Face Dataset e Tokenizar
hf_dataset = Dataset.from_dict({"text": raw_samples})

def tokenize_function(batch):
    tokens = tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False
    )
    # Em causal language modeling, os labels são iguais aos input_ids
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_dataset = hf_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
    desc="Tokenizando dataset"
)

print(f"\n✅ Dataset pronto para treino com {len(tokenized_dataset)} exemplos!")


## 🎯 7. Configuração do LoRA (PEFT)

Detecta dinamicamente os nomes das camadas lineares de qualquer arquitetura (quando configurado como `'auto'`) ou aplica os módulos indicados manualmente, acoplando os adaptadores LoRA ao modelo.


In [ ]:
import torch
from peft import LoraConfig, get_peft_model

def find_target_modules(model_to_inspect, manual_modules):
    """Detecta automaticamente os nomes de módulos lineares compatíveis com LoRA em qualquer arquitetura."""
    if manual_modules and manual_modules.strip().lower() != "auto":
        modules = [m.strip() for m in manual_modules.split(",") if m.strip()]
        print(f"ℹ️ Usando módulos LoRA especificados manualmente: {modules}")
        return modules

    linear_classes = (
        torch.nn.Linear,
    )
    try:
        from transformers.pytorch_utils import Conv1D  # usado pelo GPT-2 e similares
        linear_classes = linear_classes + (Conv1D,)
    except ImportError:
        pass
    try:
        import bitsandbytes as bnb
        linear_classes = linear_classes + (bnb.nn.Linear4bit, bnb.nn.Linear8bitLt)
    except ImportError:
        pass

    target_module_names = set()
    # Módulos que não devem ser adaptados com LoRA (saída e vocabulário)
    excluded_names = {"lm_head", "embed_tokens", "wte", "wpe", "output"}

    for name, module in model_to_inspect.named_modules():
        if isinstance(module, linear_classes):
            names = name.split(".")
            target_name = names[-1]
            if target_name not in excluded_names and not any(ex in name for ex in excluded_names):
                target_module_names.add(target_name)

    detected = sorted(list(target_module_names))
    if not detected:
        raise ValueError("❌ Nenhuma camada linear detectada automaticamente. Informe TARGET_MODULES manualmente na Célula 2.")
    print(f"🔍 Camadas lineares detectadas automaticamente ({len(detected)}): {detected}")
    return detected

print("="*60)
print("🎯 CONFIGURANDO ADAPTADOR LoRA (PEFT)...")
print("="*60)

target_modules = find_target_modules(model, TARGET_MODULES)

# GPT-2 e afins usam Conv1D (pesos transpostos) -> exige fan_in_fan_out=True
try:
    from transformers.pytorch_utils import Conv1D
    uses_conv1d = any(
        isinstance(m, Conv1D) and n.split(".")[-1] in target_modules
        for n, m in model.named_modules()
    )
except ImportError:
    uses_conv1d = False

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=target_modules,
    fan_in_fan_out=uses_conv1d,
    bias="none",
    task_type="CAUSAL_LM"
)

# Aplicar adaptadores LoRA ao modelo
lora_model = get_peft_model(model, lora_config)

# Exibir estatísticas de parâmetros treináveis
print("\n📊 ESTATÍSTICAS DE PARÂMETROS TREINÁVEIS:")
lora_model.print_trainable_parameters()
print("="*60)


## 🚀 8. Treinamento

Executa o loop de fine-tuning utilizando o `Trainer` do Hugging Face com otimizações de memória, logs periódicos e salvamento automático de checkpoints no Google Drive.


In [ ]:
import os
import time
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

print("="*60)
print("🚀 PREPARANDO O TREINAMENTO...")
print("="*60)

# Otimizador: se estiver em 4-bit/8-bit, paged_adamw_8bit reduz o consumo de memória na GPU
optimizer_choice = "paged_adamw_8bit" if QUANTIZATION in ["4-bit", "8-bit"] else "adamw_torch"

training_args = TrainingArguments(
    output_dir=str(current_checkpoint_dir),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    logging_steps=LOGGING_STEPS,
    save_strategy="steps" if SAVE_STEPS > 0 else "no",
    save_steps=SAVE_STEPS if SAVE_STEPS > 0 else 500,
    save_total_limit=2,
    fp16=env_info["use_fp16"],
    bf16=env_info["use_bf16"],
    seed=RANDOM_SEED,
    optim=optimizer_choice,
    gradient_checkpointing=GRADIENT_CHECKPOINTING,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="none"  # Desativa envio automático para ferramentas externas (ex: wandb)
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    pad_to_multiple_of=8,
    return_tensors="pt",
    padding=True
)

trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

# Verificar existência de checkpoint anterior se RESUME_TRAINING for True
checkpoint_to_resume = None
if RESUME_TRAINING:
    if current_checkpoint_dir.exists():
        checkpoints = [
            d for d in os.listdir(current_checkpoint_dir)
            if d.startswith("checkpoint-") and (current_checkpoint_dir / d).is_dir()
        ]
        if checkpoints:
            checkpoints.sort(key=lambda x: int(x.split("-")[1]))
            checkpoint_to_resume = str(current_checkpoint_dir / checkpoints[-1])
            print(f"🔄 Retomando a partir do checkpoint: {checkpoint_to_resume}")
        else:
            print(f"ℹ️ Nenhum checkpoint anterior encontrado em '{current_checkpoint_dir}'. Iniciando novo treino...")

print(f"⚡ Batch por Passo: {BATCH_SIZE} | Acumulação: {GRAD_ACCUM_STEPS} | Batch Efetivo: {BATCH_SIZE * GRAD_ACCUM_STEPS}")
print(f"📈 Épocas: {NUM_EPOCHS} | Taxa de Aprendizado: {LEARNING_RATE}")
print(f"💾 Checkpoints serão salvos em: {current_checkpoint_dir}")
print("="*60)
print("🔥 INICIANDO TREINAMENTO AGORA...\n")

start_time = time.time()
train_result = trainer.train(resume_from_checkpoint=checkpoint_to_resume)
end_time = time.time()

train_duration_sec = end_time - start_time
minutes, seconds = divmod(int(train_duration_sec), 60)
formatted_duration = f"{minutes}m {seconds}s"

final_loss = train_result.training_loss  # média de todo o treino
logged_losses = [l["loss"] for l in trainer.state.log_history if "loss" in l]
last_logged_loss = logged_losses[-1] if logged_losses else None
print("\n" + "="*60)
print(f"🎉 TREINAMENTO CONCLUÍDO COM SUCESSO!")
print(f"⏱️ Duração Total: {formatted_duration} ({train_duration_sec:.2f}s)")
print(f"📉 Loss Final de Treino: {final_loss:.4f}")
print("="*60)


## 💾 9. Salvar Adaptador LoRA e Metadados

Salva os pesos do adaptador LoRA e o tokenizer na pasta da versão (`<base>/versoes/<vN>/`) junto com o arquivo `info_treino.json` detalhando data, hiperparâmetros, loss e duração. Se o merge estiver ativado, realiza a fusão com o modelo base e salva o modelo completo.


In [ ]:
import os
import json
import datetime
import torch

print("="*60)
print(f"💾 SALVANDO ADAPTADOR LoRA EM: {current_version_dir}")
print("="*60)

# 1. Salvar Adaptador e Tokenizer
lora_model.save_pretrained(str(current_version_dir))
tokenizer.save_pretrained(str(current_version_dir))

# 2. Gerar relatório detalhado info_treino.json
info_treino = {
    "projeto": PROJECT_NAME,
    "versao": current_version_name,
    "modelo_base": MODEL_NAME_OR_PATH,
    "data_treino": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "duracao_segundos": round(train_duration_sec, 2),
    "duracao_formatada": formatted_duration,
    "total_exemplos": total_samples,
    "loss_final": round(float(final_loss), 4),
    "loss_ultimo_log": round(float(last_logged_loss), 4) if last_logged_loss is not None else None,
    "hiperparametros": {
        "num_epochs": NUM_EPOCHS,
        "learning_rate": LEARNING_RATE,
        "batch_size": BATCH_SIZE,
        "grad_accum_steps": GRAD_ACCUM_STEPS,
        "batch_efetivo": BATCH_SIZE * GRAD_ACCUM_STEPS,
        "max_length": MAX_LENGTH,
        "warmup_ratio": WARMUP_RATIO,
        "weight_decay": WEIGHT_DECAY,
        "random_seed": RANDOM_SEED
    },
    "lora_config": {
        "r": LORA_R,
        "alpha": LORA_ALPHA,
        "dropout": LORA_DROPOUT,
        "target_modules": target_modules
    },
    "hardware_ambiente": {
        "gpu": env_info["gpu_name"],
        "vram_gb": round(env_info["vram_gb"], 2),
        "quantization": QUANTIZATION,
        "precisao": "bf16" if env_info["use_bf16"] else "fp16"
    },
    "template_prompt": {
        "user_token": USER_TOKEN,
        "assistant_token": ASSISTANT_TOKEN,
        "end_token": ACTIVE_END_TOKEN,
        "system_prompt": SYSTEM_PROMPT
    }
}

info_path = current_version_dir / "info_treino.json"
with open(info_path, "w", encoding="utf-8") as f:
    json.dump(info_treino, f, indent=4, ensure_ascii=False)

print(f"✅ Adaptador salvo com sucesso em: {current_version_dir}")
print(f"📄 Metadados salvos em: {info_path}")

# 3. Fusão (Merge) opcional
if MERGE_LORA:
    print("\n" + "="*60)
    print("🔀 INICIANDO FUSÃO (MERGE) DO LoRA NO MODELO BASE...")
    print("="*60)
    merged_output_dir = current_version_dir / "modelo_completo_merged"
    os.makedirs(merged_output_dir, exist_ok=True)

    try:
        from peft import AutoPeftModelForCausalLM

        print("Carregando modelo para fusão em memória...")
        merge_dtype = torch.bfloat16 if env_info["use_bf16"] else torch.float16
        peft_merged_model = AutoPeftModelForCausalLM.from_pretrained(
            str(current_version_dir),
            torch_dtype=merge_dtype,
            device_map="auto"
        )
        print("Fundindo pesos do adaptador no modelo base...")
        full_model = peft_merged_model.merge_and_unload()

        print(f"Salvando modelo completo em: {merged_output_dir}...")
        full_model.save_pretrained(str(merged_output_dir))
        tokenizer.save_pretrained(str(merged_output_dir))
        print("✅ Fusão concluída com sucesso! Modelo completo salvo.")
    except Exception as e:
        print(f"⚠️ Erro durante a fusão: {e}")
        print("💡 Dica: O merge exige RAM de sistema livre. O adaptador LoRA continua salvo perfeitamente!")


## 💬 10. Testar o Modelo Treinado

Gera respostas interativas para testar o modelo recém-treinado, respeitando o mesmo template de prompt e os tokens de parada.


In [ ]:
# @title 🧪 Chat / Teste Rápido de Geração
# @markdown Digite o prompt de teste e ajuste os parâmetros de geração:
PROMPT_TESTE = "Olá! O que você aprendeu e como pode me ajudar?"  # @param {type:"string"}
PROMPT_SISTEMA_TESTE = ""  # @param {type:"string"}
TEMPERATURE = 0.7  # @param {type:"number"}
TOP_P = 0.9  # @param {type:"number"}
MAX_NEW_TOKENS = 256  # @param {type:"integer"}
REPETITION_PENALTY = 1.1  # @param {type:"number"}

def generate_response(prompt_text, system_text="", temp=0.7, top_p=0.9, max_tokens=256, rep_penalty=1.1):
    """Gera uma resposta formatada usando o modelo e o template configurados."""
    # Construir prompt no mesmo padrão do treinamento
    full_prompt = ""
    sys_clean = system_text.strip() or SYSTEM_PROMPT.strip()
    if sys_clean:
        full_prompt += f"{sys_clean}\n"
    full_prompt += f"{USER_TOKEN}{prompt_text.strip()}{ASSISTANT_TOKEN}"

    # Tokenizar entrada
    inputs = tokenizer(full_prompt, return_tensors="pt").to(lora_model.device)

    # Identificar IDs de parada
    stop_token_ids = [tokenizer.eos_token_id]
    if ACTIVE_END_TOKEN:
        end_ids = tokenizer.encode(ACTIVE_END_TOKEN, add_special_tokens=False)
        if end_ids:
            stop_token_ids.append(end_ids[-1])

    # Colocar modelo em modo de avaliação
    lora_model.eval()

    with torch.no_grad():
        output_ids = lora_model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=temp if temp > 0 else 1.0,
            do_sample=temp > 0,
            top_p=top_p,
            repetition_penalty=rep_penalty,
            use_cache=True,
            eos_token_id=stop_token_ids,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id
        )

    # Decodificar apenas os novos tokens gerados
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    generated_text = tokenizer.decode(new_tokens, skip_special_tokens=False)

    # Remover o token de parada do texto final exibido, se presente
    if ACTIVE_END_TOKEN and ACTIVE_END_TOKEN in generated_text:
        generated_text = generated_text.split(ACTIVE_END_TOKEN)[0]
    if tokenizer.eos_token and tokenizer.eos_token in generated_text:
        generated_text = generated_text.split(tokenizer.eos_token)[0]

    return generated_text.strip()

print("="*60)
print(f"👤 Usuário: {PROMPT_TESTE}")
print("="*60)

resposta = generate_response(
    prompt_text=PROMPT_TESTE,
    system_text=PROMPT_SISTEMA_TESTE,
    temp=TEMPERATURE,
    top_p=TOP_P,
    max_tokens=MAX_NEW_TOKENS,
    rep_penalty=REPETITION_PENALTY
)

print(f"🤖 Assistente:\n{resposta}")
print("="*60)


## 🔄 11. Carregar uma Versão Salva Anteriormente

Carrega qualquer versão de adaptador LoRA salva no Google Drive (ou a mais recente automaticamente) para realizar testes ou inferência sem precisar retreinar.


In [ ]:
# @title 📂 Carregar Adaptador de Versão Existente
# @markdown Escolha a versão para carregar ("auto" seleciona a mais recente):
VERSAO_PARA_CARREGAR = "auto"  # @param {type:"string"}

import os
import re
import json
from pathlib import Path
from peft import PeftModel

def load_saved_version(version_target):
    """Carrega adaptador LoRA salvo na pasta de versões do projeto."""
    versions_path = Path(BASE_DIR) / "versoes"

    if not versions_path.exists():
        raise FileNotFoundError(f"❌ A pasta de versões não existe em: {versions_path}")

    if version_target.strip().lower() == "auto":
        existing = [
            d for d in os.listdir(versions_path)
            if (versions_path / d).is_dir() and re.match(r"^v(\d+)$", d)
        ]
        if not existing:
            all_dirs = [d for d in os.listdir(versions_path) if (versions_path / d).is_dir()]
            if not all_dirs:
                raise FileNotFoundError(f"❌ Nenhuma versão encontrada dentro de {versions_path}!")
            selected_version = sorted(all_dirs)[-1]
        else:
            indices = [int(re.match(r"^v(\d+)$", d).group(1)) for d in existing]
            selected_version = f"v{max(indices)}"
    else:
        selected_version = version_target.strip()

    target_dir = versions_path / selected_version
    if not target_dir.exists():
        raise FileNotFoundError(f"❌ Versão '{selected_version}' não foi encontrada em {versions_path}!")

    print(f"📦 Carregando versão: {selected_version} de {target_dir}...")

    existing = globals().get("lora_model")
    if isinstance(existing, PeftModel):
        # LoRA já acoplado nesta sessão: carrega como novo adaptador (evita acoplar duas vezes)
        if selected_version not in existing.peft_config:
            existing.load_adapter(str(target_dir), adapter_name=selected_version)
        existing.set_adapter(selected_version)
        loaded_model = existing
    else:
        # Sessão nova (células 1-5 executadas): acopla ao modelo base limpo
        loaded_model = PeftModel.from_pretrained(model, str(target_dir))
    loaded_model.eval()

    # Ler info_treino.json se existir
    info_file = target_dir / "info_treino.json"
    if info_file.exists():
        with open(info_file, "r", encoding="utf-8") as f:
            meta = json.load(f)
            print("\n📋 Resumo da Versão Carregada:")
            print(f"  • Data do Treino:  {meta.get('data_treino')}")
            print(f"  • Duração:         {meta.get('duracao_formatada')}")
            print(f"  • Exemplos:        {meta.get('total_exemplos')}")
            print(f"  • Loss Final:      {meta.get('loss_final')}")

    print(f"\n✅ Versão '{selected_version}' carregada com sucesso e pronta para uso!")
    return loaded_model

lora_model = load_saved_version(VERSAO_PARA_CARREGAR)


## 🧹 12. Limpeza de Memória GPU (Opcional)

Libera caches de VRAM da GPU e executa a coleta de lixo do Python para liberar recursos quando desejar.


In [ ]:
# @title 🧹 Liberar Memória VRAM
import gc
import torch

def clean_gpu_memory():
    """Libera caches de VRAM e executa garbage collection."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        allocated = torch.cuda.memory_allocated() / (1024 ** 2)
        reserved = torch.cuda.memory_reserved() / (1024 ** 2)
        print("="*50)
        print("🧹 Memória de GPU liberada com sucesso!")
        print(f"💾 VRAM Alocada Atual:  {allocated:.2f} MB")
        print(f"💾 VRAM Reservada Atual: {reserved:.2f} MB")
        print("="*50)

clean_gpu_memory()
